**PROYECTO**       : MIGRACION SANDBOX  
**NOMBRE**         : ddl_td_pago_servicio.ipynb  
**TABLA DESTINO**  : mb_gold_prod.inof.td_pago_servicio  
**TABLA FUENTE**   : NA  
**OBJETIVO**       : Creacion de la tabla de pagos de servicio en la capa gold  
**TIPO**           : PYTHON  
**REPROCESABLE**   : SI  
**OBSERVACION**    : NA  
**SCHEDULER**      : NA  
**JOB**            : NA
| VERSION | DESARROLLADOR | PROVEEDOR | PO | FECHA | DESCRIPCION |
|---------|---------------|-----------|----|-------|-------------|
| 1.0 | Esteban Offer | MIBANCO | Willyan Soto | 2026-09-11 | Creacion de tabla |

## 1. Parametros de entrada

In [0]:
dbutils.widgets.text("ambiente", "dev")
var_ambiente = dbutils.widgets.get("ambiente")

import logging
import time

logger = logging.getLogger("DDL_TD_PAGO_SERVICIO")
logger.setLevel(logging.INFO)
ini_proceso = time.perf_counter()
logger.info("Inicio de la creacion de tabla. ambiente=%s", var_ambiente)

## 2. Constantes y variables

In [0]:
TBL_PAGO_FIN = f"mb_gold_{var_ambiente}.inof.td_pago_servicio"

## 3. Creacion de la tabla

In [0]:
ddl_pago_servicio = f"""
CREATE TABLE IF NOT EXISTS {TBL_PAGO_FIN} (
    cod_pago_servicio          STRING         COMMENT 'Codigo unico del pago de servicio',
    fec_pago_servicio          TIMESTAMP      COMMENT 'Fecha en que se registro el pago',
    nom_empresa_servicio       STRING         COMMENT 'Nombre de la empresa prestadora',
    mto_servicio_origen        DECIMAL(18,2)  COMMENT 'Monto del servicio en moneda origen',
    nro_documento_cliente_dac  STRING         COMMENT 'Numero de documento del cliente',
    _ingestion_time            TIMESTAMP      COMMENT 'Marca de ingesta del registro',
    _processing_time           TIMESTAMP      COMMENT 'Marca de procesamiento del registro'
)
USING DELTA
COMMENT 'Pagos de servicio registrados por Data Entry'
PARTITIONED BY (fec_pago_servicio)
TBLPROPERTIES (
    'frecuencia' = 'diaria',
    'naturaleza' = 'transaccional',
    'tipo_tabla' = 'DataEntry',
    'owner'      = 'inof',
    'dac'        = 'si',
    'input'      = 'DATAENTRY'
)
"""

try:
    spark.sql(ddl_pago_servicio)
except Exception as exc:
    logger.error("Fallo la creacion de %s: %s", TBL_PAGO_FIN, exc)
    raise

logger.info("Fin de la creacion. Duracion total: %.2f segundos",
            time.perf_counter() - ini_proceso)